# Teaching NanoGPT to Do Math with DPO: working out the steps

SC3000/CZ3005 Assignment 1

**Team and contributions:** *(to fill in before submission: team name, members, and what each member did.)*

## Overview

The model is a small character-level NanoGPT (8.8M parameters) pretrained on question-answer chat data. Before fine-tuning it answers arithmetic questions with "Sorry, I don't know." We fine-tune it with DPO (Direct Preference Optimization): for each question the training data holds a preferred reply (the *positive*) and a rejected reply (the *negative*), and the loss makes the positive more likely than the negative.

In our first attempt, the positives stated the answer first, like the sample file: `86*83=? The answer is 7138 because 86*83 equals 7138.` That model got all additions and subtractions among our test questions (the same 600 as in this notebook) right and 80% of the divisions, but only a third of the multiplications, because it had to write all digits of a product in one go. This notebook changes two things:

1. **The positive works the answer out in steps** before stating it: `86*83=? 86*80 is 6880, 86*3 is 258, 6880+258 is 7138, so the answer is 7138.` Each step is an easier calculation, and the model can read its earlier results while writing the next one (Task 1, following Lee et al., 2024).
2. **The model's own mistakes become negatives.** The Task 1 negatives are refusals, which the model stops producing early in training. Between epochs the model answers some training questions, and each wrong reply is added as a new negative for that question (Step 7).

| Task | Where |
|---|---|
| Task 1: build positive-negative pairs | the "Task 1" section after Step 4 |
| Task 2: fine-tune with DPO | Steps 5–7 |
| Task 3: test the fine-tuned model | Step 8 |
| Task 4: explanation and analysis | the text cells |

**How to run.** Start Jupyter from the folder that holds this notebook, because the code uses relative paths such as `../sft/gpt.pt` and `../model.py`, and run the cells from top to bottom.

**To test the submitted `dpo.pt` only** (without rebuilding the data or retraining): run Steps 1–4 and the Task 1 code cells except the last one (the one that calls `main`), skip Steps 5–7, then run Step 8.

**Changes to the provided template** (each one is explained where it happens):
- Step 1 installs packages with `%pip` instead of `!pip`.
- Step 2 also uses Apple's GPU (`mps`), drops the `CUDA_VISIBLE_DEVICES = "1"` line, sets `max_length = 112`, uses learning rate `1e-3` and 12 epochs instead of `1e-4` and 5, and adds one setting, `questions_to_check`.
- Step 6 uses AdamW and lowers the learning rate along a cosine curve, once per epoch.
- Step 7 writes the loss as its two parts, clips the gradients, checks the accuracy on the test questions after every epoch, adds the model's mistakes as new pairs between epochs, and saves all pairs back to `pos_neg_pairs.json` at the end.
- Step 8 completes the test loop, uses `.to(device)` instead of `.cuda()` and loads `./dpo.pt`, the file Step 7 saves.

### Step 1: Install necessary packages

`%pip` installs into the environment of the running kernel. `!pip` runs whichever `pip` comes first on the shell's PATH, which can belong to a different Python. Only `torch`, `matplotlib` and `tqdm` are imported below; the other packages are kept from the template.

In [ ]:
%pip install matplotlib
%pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

Changes from the template:
- **Device.** The template only checks for CUDA. We also use Apple's GPU (`mps`) when there is one, and fall back to the CPU otherwise.
- **`CUDA_VISIBLE_DEVICES = "1"` removed.** That line hides every GPU except the second one. On a machine with a single GPU, PyTorch would then see no GPU at all and silently train on the CPU.
- **`max_length = 112` instead of 64.** The steps make the positives longer: the longest is 102 characters with the 4 newlines (printed in Task 1), and 112 leaves a little room for the wrong replies that Step 7 adds as negatives. `pad_or_truncate` keeps the *last* `max_length` characters, so a longer sample loses its start, the question.
- **Learning rate `1e-3` instead of `1e-4`, and 12 epochs instead of 5.** In a separate one-epoch trial (not shown here), the template's `1e-4` taught the model the sentence pattern, but only 1.7% of its answers to our test questions were right (4% with `1e-3`); we did not try `1e-4` for longer. With `1e-3` and 12 epochs, our first attempt's accuracy stayed low for two epochs and then jumped (3.0%, 7.3% and 35.3% after epochs 1–3), so we keep these settings.
- **`questions_to_check = 2000`** is new: the number of training questions the model answers between epochs to find its mistakes (Step 7).
- `re` and `Counter` are imported for Task 1 and Step 8.

The other settings are the template's: `beta = 0.5`, batch size 64, and the sampling settings for `gpt.generate` (temperature 0.8, top_k 200, at most 200 new characters).

The tokenizer works on single characters: `meta.pkl` maps each of the 74 characters to an id. The newline character has id 0, which is also used for padding, and `.` has id 7. `gpt.generate` in `model.py` stops at either of these two.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
from tqdm import tqdm
import time
import json
import re
from collections import Counter
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print("device: ", device)
base_lr = 1e-3
epochs = 12
batch_size = 64
max_length = 112
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
questions_to_check = 2000      # training questions the model answers between epochs, to find its mistakes
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

These three functions come with the template and are used unchanged:
- `compute_logprob(input_ids)` runs the model on each sequence and returns the **average log-probability per character**, leaving out padding (id 0). For a batch of `B` sequences it returns a tensor of shape `B`.
- `pad_or_truncate(seq, max_length)` pads a sequence with id 0 up to `max_length`. A longer sequence keeps only its last `max_length` ids.
- `get_batches(lines, batch_size)` shuffles the pairs, appends 4 newlines to each negative and positive, encodes and pads them, and yields `(neg_tensor, pos_tensor)`, each of shape `batch_size × max_length`. The last incomplete batch is dropped.

Because the newline is id 0, the 4 newlines appended in `get_batches` are treated as padding and never scored. The model therefore learns to end a reply with `.`, which is also where `gpt.generate` stops.

In [ ]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

This loads `sft/gpt.pt`: 6 layers, 6 attention heads, 348-dimensional embeddings, a context of 256 characters and a 74-character vocabulary, 8.8M parameters in total. The `_orig_mod.` prefix that `torch.compile` adds to the weight names is removed before loading. The model is left in training mode, so dropout (p = 0.2) is on.

In [ ]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

### Task 1: Build positive-negative data pairs

Task 1 has four code cells: the questions, the positives, the functions that let the model answer a question and check its reply, and the negatives. The last one writes all pairs to `./pos_neg_pairs.json`; Step 7 later adds the model's mistakes to this file.

**Questions.** These are the same questions as in our first attempt.
- There are 12 types: the four operations asked directly (`a+b=?`, `a-b=?`, `a*b=?`, `a/b=?`), and each operation as an equation with `x` on the left (`x+a=b,x=?`) or on the right (`a+x=b,x=?`). The format copies the Step 8 test set exactly, with no space after the comma, because the model reads one character at a time.
- For each type we take every pair of numbers from 0–100 and work out the third number from them, so the answer is always a whole number; for `a/b=?` the pair is `b` and the answer, and `a = b*answer`. A pair is skipped if it would divide by 0, need a negative number (as in `5-8=?` or `5-x=-3,x=?`), or give an equation with more than one solution (such as `0*x=0`).
- The questions of each type are shuffled. The first 50 are kept aside as **test questions**, which are never trained on; the next 9000 become training questions. Only `a-b=?` and `a-x=b,x=?` have fewer questions (5151 each, so 5101 for training), which gives 100,202 training questions.
- The 600 test questions are our own, not the template's test set in Step 8; the five questions of that test set are among the training questions. A test question is never trained on as a question, but its calculation usually is, in another form: the steps for `24*14=?` are also in the positive for `x/14=24,x=?`. So the test accuracy shows whether the model turns a question it has not seen into the right calculation and carries it out, not whether it can do arithmetic it has never seen.

In [ ]:
random.seed(0)
torch.manual_seed(0)
numbers = range(0, 101)
train_per_type = 9000          # training questions per type (a-b=? and a-x=b,x=? have fewer)
test_per_type = 50             # questions per type kept out of training, used as test questions
data_path = "./pos_neg_pairs.json"

# The 12 question types. Each function takes two numbers drawn from 0..100 and returns
# (question, answer, calculation), where `calculation` is what has to be worked out to get the answer,
# or None when the pair is not allowed.
def add(a, b):    return f"{a}+{b}=?", a + b, f"{a}+{b}"
def sub(a, b):    return (f"{a}-{b}=?", a - b, f"{a}-{b}") if b <= a else None
def mul(a, b):    return f"{a}*{b}=?", a * b, f"{a}*{b}"
def div(b, q):    return (f"{b*q}/{b}=?", q, f"{b*q}/{b}") if b > 0 else None

def x_add(x, a):  return f"x+{a}={x+a},x=?", x, f"{x+a}-{a}"
def add_x(x, a):  return f"{a}+x={a+x},x=?", x, f"{a+x}-{a}"
def x_sub(a, b):  return f"x-{a}={b},x=?", a + b, f"{b}+{a}"
def sub_x(a, x):  return (f"{a}-x={a-x},x=?", x, f"{a}-{a-x}") if x <= a else None
def x_mul(x, a):  return (f"x*{a}={x*a},x=?", x, f"{x*a}/{a}") if a > 0 else None
def mul_x(x, a):  return (f"{a}*x={a*x},x=?", x, f"{a*x}/{a}") if a > 0 else None
def x_div(a, b):  return (f"x/{a}={b},x=?", a * b, f"{b}*{a}") if a > 0 else None
def div_x(x, b):  return (f"{x*b}/x={b},x=?", x, f"{x*b}/{b}") if x > 0 and b > 0 else None

question_types = [add, sub, mul, div, x_add, add_x, x_sub, sub_x, x_mul, mul_x, x_div, div_x]

def build_questions(make):
    """Collect every allowed question of one type, in random order."""
    items = []
    for p in numbers:
        for q in numbers:
            item = make(p, q)
            if item is not None:
                items.append(item)
    random.shuffle(items)
    return items

train_items = []               # (question, answer, calculation)
test_items = []                # the same, but never used for training
for make in question_types:
    items = build_questions(make)
    test_items += items[:test_per_type]
    train_items += items[test_per_type:test_per_type + train_per_type]
print("Number of training questions:", len(train_items))
print("Number of test questions:", len(test_items))

**Positives: the working comes before the answer.** Each question function also returns the calculation that gives the answer (for `x/86=83,x=?` it is `83*86`), and `write_positive` writes it out in steps:

| Question | Positive (after the question) |
|---|---|
| `86*83=?` | `86*80 is 6880, 86*3 is 258, 6880+258 is 7138, so the answer is 7138.` |
| `7138/86=?` | `86*80 is 6880, 7138-6880 is 258, 86*3 is 258, so the answer is 83.` |
| `x/86=83,x=?` | `x is 83*86, 83*80 is 6640, 83*6 is 498, 6640+498 is 7138, so the answer is 7138.` |
| `x+55=95,x=?` | `x is 95-55, 95-55 is 40, so the answer is 40.` |

- **Multiplication:** multiply by the tens and by the ones of the second number separately, then add the two results. If the second number is a single digit or a multiple of 10, it takes one step.
- **Division:** as in long division, first the tens of the answer (86×80 = 6880 still fits into 7138), then the ones from what is left (258 = 86×3). If the answer is a single digit or a multiple of 10, it takes one step (`100*90 is 9000`).
- **Equations:** first say which calculation gives x, then work it out.
- **Addition and subtraction:** one step, since our first attempt already got all of these right.

Lee et al. (2024) found that small transformers learn arithmetic much better when the training text shows the intermediate steps. Because `gpt.generate` stops at `.`, the steps are separated by commas and only the last sentence ends with a `.`. The reply always ends with "the answer is N", so `reply_is_correct` can find the answer. The cell prints the positive for one test question of each type and the length of the longest positive.

In [ ]:
def multiply_steps(a, b):
    """Steps for a*b: multiply a by the tens and by the ones of b, then add the two results."""
    tens, ones = b // 10 * 10, b % 10        # 83 -> 80 and 3
    if tens == 0 or ones == 0:               # b is a single digit, a multiple of 10, or 100
        return [f"{a}*{b} is {a * b}"]
    return [f"{a}*{tens} is {a * tens}",
            f"{a}*{ones} is {a * ones}",
            f"{a * tens}+{a * ones} is {a * b}"]

def divide_steps(a, b):
    """Steps for a/b, which always divides exactly:
    find the tens of the answer, then the ones from what is left."""
    answer = a // b
    tens, ones = answer // 10 * 10, answer % 10
    if tens == 0 or ones == 0:
        return [f"{b}*{answer} is {a}"]
    rest = a - b * tens
    return [f"{b}*{tens} is {b * tens}",
            f"{a}-{b * tens} is {rest}",
            f"{b}*{ones} is {rest}"]

def write_positive(question, answer, calculation):
    """The positive reply: the question, the steps of the calculation, and the answer."""
    a, sign, b = re.fullmatch(r"(\d+)(\D)(\d+)", calculation).groups()     # "83*86" -> "83", "*", "86"
    if sign == "*":
        steps = multiply_steps(int(a), int(b))
    elif sign == "/":
        steps = divide_steps(int(a), int(b))
    else:                                    # addition and subtraction take one step
        steps = [f"{calculation} is {answer}"]
    if "x" in question:                      # first say which calculation gives x
        steps = [f"x is {calculation}"] + steps
    # gpt.generate stops at ".", so the steps are joined with commas and only the end has a "."
    return f"{question} {', '.join(steps)}, so the answer is {answer}."

questions = [question for question, answer, calculation in train_items]
positives = [write_positive(question, answer, calculation) for question, answer, calculation in train_items]
test_questions = [question for question, answer, calculation in test_items]

print("The positive for the first test question of each type:")
for question, answer, calculation in test_items[::test_per_type]:
    print(" ", write_positive(question, answer, calculation))
longest = max(len(positive) for positive in positives) + 4      # get_batches adds 4 newlines
print(f"Longest positive: {longest} characters with the 4 newlines (max_length = {max_length})")

**Answering and checking.** `generate_reply` lets the model answer one question with `gpt.generate` and the template's sampling settings. `reply_is_correct` reads the number after "the answer is" in a reply, puts it back into the question and checks the arithmetic: for `x+55=95,x=?` and the answer 40, it checks that 40+55 equals 95. Only the final answer is checked, not the steps. `test_accuracy` uses both to count how many test questions the model gets right. These functions are used again in Steps 7 and 8.

In [ ]:
def generate_reply(question):
    """The model's reply to one question, sampled with gpt.generate.
    The returned text starts with the question."""
    x = torch.tensor([encode(question)], dtype=torch.long, device=device)
    with torch.no_grad():                    # sampling only, no gradients needed
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    return decode(y[0].tolist())

def reply_is_correct(question, reply):
    """Put the number after "the answer is" in the reply back into the question and check the arithmetic."""
    # the reply starts with the question, so only search the part after it
    match = re.search(r"the answer is (\d+)", reply[len(question):])
    if match is None:
        return False
    answer = int(match.group(1))                         # int, so a reply like "06" becomes 6
    if "x" in question:                                  # "x+55=95,x=?" -> 40+55 == 95
        left, right = question.split(",")[0].replace("x", str(answer)).split("=")
        try:
            return eval(left) == eval(right)
        except ZeroDivisionError:                        # answering 0 to "84/x=7,x=?"
            return False
    return eval(question[:-2]) == answer                 # "79-7=?" -> 79-7 == 72

def test_accuracy():
    """Answer the test questions with the current model.
    Returns the fraction answered correctly and each question's reply."""
    gpt.eval()                                           # no dropout while answering
    reply_of = {}
    for question in test_questions:
        reply_of[question] = generate_reply(question)
    gpt.train()
    correct = 0
    for question in test_questions:
        if reply_is_correct(question, reply_of[question]):
            correct += 1
    return correct / len(test_questions), reply_of

**Negatives.** `main` lets the pretrained model answer every training question, and its reply becomes the negative: a refusal such as "Sorry, I don't know." Then it saves all pairs to `pos_neg_pairs.json`. This is the slow part of Task 1. Step 7 later adds a second kind of negative: the fine-tuned model's own wrong answers.

**Reproducibility.** The seed makes the questions and positives the same on every machine. The negatives also depend on the device's random number generator, so running this cell on another device gives slightly different negatives and overwrites `pos_neg_pairs.json`.

In [ ]:
def main(questions, positives, path):
    """Let the pretrained model answer every question, then save the (negative, positive) pairs to `path`."""
    start = time.time()
    gpt.eval()                    # no dropout while sampling
    negatives = []
    for question in tqdm(questions):
        negatives.append(generate_reply(question))
    gpt.train()                   # back to training mode for Step 7
    print(f"Generated {len(negatives)} negatives in {(time.time() - start) / 60:.1f} minutes")

    pairs = []
    for negative, positive in zip(negatives, positives):
        pairs.append({"negative": negative, "positive": positive})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=1)
    print(f"Saved {len(pairs)} pairs to {path}")
    print("Some example pairs:")
    for pair in random.sample(pairs, 5):
        print(pair)

# Generate the negatives with the pretrained model and save all pairs; this is the slow part of Task 1.
# To test the submitted dpo.pt without rebuilding the data, skip this cell (see "How to run" at the top).
main(questions, positives, data_path)

### Step 5: Load Data

We load the file written by Task 1 (`data_path`). The template comment says `./data/pos_neg_pairs.json`, but there is no `data/` folder. The list keeps the template's name, `lines`.

In [ ]:
# Load the pairs built in Task 1
with open(data_path, encoding="utf-8") as f:
    lines = json.load(f)
print("Number of pairs:", len(lines))
print("Example pair:", lines[0])

### Step 6: Build the optimizer and scheduler

- **AdamW**, as the template recommends, with the learning rate `base_lr = 1e-3` from Step 2 and weight decay 0.01 (AdamW's default).
- **Cosine schedule.** The learning rate starts at `1e-3` and falls along a cosine curve towards `1e-4`. We start with a large learning rate so that the early epochs learn quickly, and lower it so that the later epochs only make small changes. Step 7 adds pairs during training, so the number of batches per epoch is not known in advance; we therefore lower the learning rate once per epoch (`scheduler.step()` at the end of each epoch) instead of once per batch.

In [ ]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)

# The learning rate goes from base_lr towards base_lr/10 along a cosine curve, one step per epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=base_lr / 10)

### Step 7: Begin training

**Loss.** We use the loss suggested in the template:

$$\text{loss} = -\log\sigma\!\left(\frac{\log p(\text{pos}) - \log p(\text{neg})}{\beta}\right) - 0.1\,\log p(\text{pos})$$

The first term is a DPO-style preference loss and the second an SFT (supervised fine-tuning) loss on the positive. Unlike the DPO paper, there is no frozen reference model: both log-probabilities come from the model being trained. Here $\log p(\cdot)$ is the average log-probability per character of a whole sequence, computed by `compute_logprob` from Step 3, and $\beta = 0.5$. The code computes the two terms separately so that we can follow the DPO term on its own.

**The model's own mistakes as new negatives.** The Task 1 negatives are refusals. Once the model has stopped refusing, the positive is already far more likely than such a negative, the DPO term is close to 0, and only the SFT term is still teaching. So after every epoch except the last (mistakes found then would never be trained on), `collect_mistakes` lets the model answer `questions_to_check` = 2000 random training questions, and each wrong reply becomes a new pair: the wrong reply is the negative and the worked positive for the same question is the positive. Such a negative looks like the positive up to the first wrong step, so the DPO term mostly pushes down the reply from the first wrong step on. This is a simpler version of Pang et al. (2024), who train in rounds on pairs of the model's own correct and wrong answers and also add an SFT term like ours; our positives are always the worked positives from Task 1.

**Saving the pairs.** The new pairs stay in `lines` for the rest of training. At the end, all pairs are saved back to `pos_neg_pairs.json`, so the submitted file holds all the training data. If Steps 5–7 are run again without the Task 1 cell that calls `main`, they also train on these saved mistakes.

**Test after every epoch.** The loss only shows how closely the model copies the training replies, not whether it can answer questions it was not trained on. So after every epoch the model answers the 600 test questions from Task 1 and we print how many it gets right, along with one example reply for six of the question types.

**Other details.** Gradients are clipped to a norm of 1.0, so a single unusual batch cannot move the model too far. The checkpoint `dpo.pt` is saved after every epoch, as in the template.

In [ ]:
def collect_mistakes(count):
    """Let the model answer `count` random training questions. Each wrong reply becomes a new pair:
    the wrong reply as the negative, the worked positive for the same question as the positive."""
    gpt.eval()
    new_pairs = []
    for i in random.sample(range(len(questions)), count):
        reply = generate_reply(questions[i])
        if not reply_is_correct(questions[i], reply):
            new_pairs.append({"negative": reply, "positive": positives[i]})
    gpt.train()
    return new_pairs

In [ ]:
loss_history = []
dpo_loss_history = []          # the DPO term of the loss on its own
accuracy_history = []          # accuracy on the test questions after each epoch
start = time.time()
for epoch in range(epochs):
    total_steps = len(lines) // batch_size     # batches in this epoch; grows as mistakes are added
    pbar = tqdm(get_batches(lines, batch_size), total=total_steps)
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)
        # DPO term: make the positive more likely than the negative
        dpo_loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean()
        # SFT term: also raise the probability of the positive directly
        sft_loss = -pos_logprob.mean() * 0.1
        loss = dpo_loss + sft_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), 1.0)
        optimizer.step()

        loss_history.append(loss.item())
        dpo_loss_history.append(dpo_loss.item())
        pbar.set_description(f"epoch {epoch + 1}/{epochs}, loss {loss.item():.3f}")
    scheduler.step()
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    epoch_loss = sum(loss_history[-total_steps:]) / total_steps
    epoch_dpo_loss = sum(dpo_loss_history[-total_steps:]) / total_steps
    accuracy, reply_of = test_accuracy()
    accuracy_history.append(accuracy)
    print(f"Epoch {epoch + 1}: average loss {epoch_loss:.4f} (DPO term {epoch_dpo_loss:.4f}), "
          f"test accuracy {accuracy:.1%}. Saved checkpoint to {ckpt_path}")
    # one test reply each for a+b, a*b, x+a=b, x-a=b, x*a=b, x/a=b
    for question in test_questions[::2 * test_per_type]:
        print("   ", reply_of[question])

    if epoch < epochs - 1:                   # mistakes found after the last epoch would never be trained on
        new_pairs = collect_mistakes(questions_to_check)
        lines += new_pairs
        print(f"    {len(new_pairs)} of {questions_to_check} training questions answered wrongly; added as new pairs")
        if new_pairs:
            print("    for example:", new_pairs[0]["negative"])

# Save all pairs, including the ones added during training
with open(data_path, "w", encoding="utf-8") as f:
    json.dump(lines, f, ensure_ascii=False, indent=1)
print(f"Saved {len(lines)} pairs to {data_path}")
print(f"Training took {(time.time() - start) / 60:.1f} minutes")

#### Training curve

*(The description of the curves is written after the full run.)*

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(loss_history, linewidth=0.8, label="total loss")
ax_loss.plot(dpo_loss_history, linewidth=0.8, label="DPO term")
ax_loss.set_yscale("log")                      # the loss spans a wide range
ax_loss.set_xlabel("training step")
ax_loss.set_ylabel("loss (log scale)")
ax_loss.set_title("Training loss")
ax_loss.legend()
epochs_done = range(1, len(accuracy_history) + 1)
ax_acc.plot(epochs_done, [100 * a for a in accuracy_history], linewidth=2, marker="o", markersize=7)
ax_acc.set_xlabel("epoch")
ax_acc.set_ylabel("correct answers (%)")
ax_acc.set_ylim(0, 100)
ax_acc.set_title(f"Accuracy on the {len(test_questions)} test questions")
for ax in (ax_loss, ax_acc):
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Step 8: Begin testing

Step 7 already checked the model on the test questions after every epoch, to follow the training. Step 8 tests the final result the way Task 3 asks: it loads the saved `dpo.pt` into a new model and answers the template's test set. We completed the test loop and changed two things in the template code: `.to(device)` instead of `.cuda()`, so it also runs on a Mac or a CPU, and the path `./dpo.pt`, where Step 7 saves the model. We print the model's reply to each question.

In [ ]:
# Load the fine-tuned model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))

*(Comments on these replies: written after the full run.)*

#### Accuracy on the test questions, by question type

Here the final model answers the 600 test questions once more, with the same sampling as above, and we count the correct answers for each question type. A reply is an *exact match* when the whole reply, every step included, is the positive we would have written for that question; a reply can have the right answer without being an exact match. The replies are sampled again, so the total can differ a little from the last epoch in Step 7.

In [ ]:
def question_type(question):
    return re.sub(r"\d+", "n", question)          # "x+55=95,x=?" -> "x+n=n,x=?"

_, reply_of = test_accuracy()

correct_by_type = Counter()
total_by_type = Counter()
exact_matches = 0
wrong_replies = []
for question, answer, calculation in test_items:
    reply = reply_of[question]
    kind = question_type(question)
    total_by_type[kind] += 1
    if reply_is_correct(question, reply):
        correct_by_type[kind] += 1
    else:
        wrong_replies.append(reply)
    if reply == write_positive(question, answer, calculation):
        exact_matches += 1

print(f"{'question type':<14} correct")
for kind in total_by_type:
    print(f"{kind:<14} {correct_by_type[kind]:3d} / {total_by_type[kind]}")
total_correct = sum(correct_by_type.values())
print(f"{'all':<14} {total_correct:3d} / {len(test_items)} = {total_correct / len(test_items):.1%}")
print(f"Exact matches (answer and every step): {exact_matches} / {len(test_items)}")
print("Some wrong replies:")
for reply in wrong_replies[:8]:
    print(" ", reply)

*(Analysis by question type: written after the full run.)*

## Summary

*(Written after the full run.)*

## References

- A. Karpathy, *nanoGPT*, https://github.com/karpathy/nanoGPT. The course's `model.py` and `configurator.py` are adapted from this repository (for example, `generate` stops at a newline or `.`); we did not change them.
- Course base code, https://github.com/zhouyuan888888/NanoGPT-Math. It provides the pretrained model `sft/gpt.pt`, the character codebook `sft/meta.pkl`, the sample `pos_neg_pairs.json`, and the template notebook: the code of Steps 1–4 (Step 2 with the changes listed there), `compute_logprob`, `pad_or_truncate`, `get_batches`, the skeletons of Steps 7 and 8, and the suggested loss.
- R. Rafailov, A. Sharma, E. Mitchell, S. Ermon, C. D. Manning, C. Finn, "Direct Preference Optimization: Your Language Model is Secretly a Reward Model", NeurIPS 2023, https://arxiv.org/abs/2305.18290.
- N. Lee, K. Sreenivasan, J. D. Lee, K. Lee, D. Papailiopoulos, "Teaching Arithmetic to Small Transformers", ICLR 2024, https://arxiv.org/abs/2307.03381.
- R. Y. Pang, W. Yuan, K. Cho, H. He, S. Sukhbaatar, J. Weston, "Iterative Reasoning Preference Optimization", NeurIPS 2024, https://arxiv.org/abs/2404.19733.
- PyTorch documentation for `torch.optim.AdamW`, `torch.optim.lr_scheduler.CosineAnnealingLR` and `torch.nn.utils.clip_grad_norm_`.

Written for this assignment: the Task 1 cells, the code of Steps 5–7 (including `collect_mistakes`), the completion of the Step 8 test loop and the evaluation by question type, and all explanations.